# pcap2mtam

- Read and process a (large) number of PCAP files, creating an MTAM matrix per file.
- Flatten and save the MTAMs as well as the maching backend

In [ ]:
from pathlib import Path
import pcap_tools as pt
import re
import numpy as np
from tqdm import tqdm

AGENT_TYPE = "weather_agent"  # weather_agent, no-tools_agent
SET_TYPE = "train"  # train, test
MAX_MATRIX_LEN = 600 # 4800 for research_agent, 600 for weather_agent and no-tools_agent
PCAP_DIR = f"/Users/tarik/data/{AGENT_TYPE}_{SET_TYPE}/pcap/"
CLIENT_IP = "172.17.0.2"

In [ ]:
# Helpers

pcap_file_name_pattern = re.compile(r'^([^-\n]+)-(.+)-(\d{8}-\d{6})\.pcap$')

def extract_values(filename):
    m = pcap_file_name_pattern.match(filename)
    if not m:
        raise ValueError(f"Invalid filename format: {filename}")
    value1, value2, value3 = m.groups()
    return value1, value2, value3

BACKEND_CODES = {
    "openai": 0,
    "gemini": 1,
    "deepseek": 2,
    "ollama": 3,
}
def get_code(backend):
    return BACKEND_CODES.get(backend, -1)  # Return -1 for unknown backends

In [ ]:
files = [p for p in Path(PCAP_DIR).iterdir() 
         if p.is_file() and p.suffix.lower() in (".pcap", ".pcapng")]

mtams = []
targets = []

for p in tqdm(files, desc="Processing PCAPs"):
    backend, model, dt = extract_values(p.name)
    trace = pt.pcap_to_trace_scapy(str(p), CLIENT_IP)

    mtam = pt.build_mtam(trace, window_size=0.1, num_windows=MAX_MATRIX_LEN)
    mtams.append(mtam.reshape(-1))
    targets.append(get_code(backend))

In [ ]:
# Create and check numpy arrays

X = np.vstack(mtams)
y = np.array(targets)
assert X.shape[0] == y.shape[0]
assert np.all(y >= 0)

data_dict = {'dataset': [], 'label': []}
data_dict['dataset'], data_dict['label'] = X, y
# Save to disk

np.save(f"../ml/datasets/{AGENT_TYPE}_{SET_TYPE}.npy", data_dict)
# np.save("X1.npy", X)
# np.save("y1.npy", y)